In [1]:
!git clone https://github.com/pals-ucb/privacy-ner-att.git



fatal: destination path 'privacy-ner-att' already exists and is not an empty directory.


In [2]:
!pip install -r privacy-ner-att/requirements.txt

  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached accelerate-1.4.0-py3-none-any.whl.metadata (19 kB)
  Using cached aiohappyeyeballs-2.5.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiohttp-3.11.13-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.7 kB)
  Using cached anyio-4.8.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached appnope-0.1.4-py2.py3-none-any.whl.metadata (908 bytes)
  Using cached arrow-1.3.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached async_lru-2.0.4-py3-none-any.whl.metadata (4.5 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached attrs-25.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached datasets-3.3.2-py3-none-any.whl.metadata (19 kB)
  Using cached debugpy-1.8.13-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.3 kB)
  Using cached filelock-3.17.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached fontt

In [3]:
pip install datasets

In [4]:
!cd privacy-ner-att/datasets/; unzip hf_privacy_ner_cleaned.zip

Archive:  hf_privacy_ner_cleaned.zip
replace hf_privacy_ner_cleaned/dataset_dict.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace hf_privacy_ner_cleaned/test/state.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: hf_privacy_ner_cleaned/test/state.json  
  inflating: hf_privacy_ner_cleaned/test/cache-0bb6367ca4f1e00a.arrow  
  inflating: hf_privacy_ner_cleaned/test/cache-d0ed1cd06ce6f914.arrow  
  inflating: hf_privacy_ner_cleaned/test/dataset_info.json  
  inflating: hf_privacy_ner_cleaned/test/data-00000-of-00001.arrow  
  inflating: hf_privacy_ner_cleaned/test/cache-e4faa0f52c4ad25f.arrow  
  inflating: hf_privacy_ner_cleaned/train/state.json  
  inflating: hf_privacy_ner_cleaned/train/cache-26a4635317525fea.arrow  
  inflating: hf_privacy_ner_cleaned/train/cache-01e42c1b16071b32.arrow  
  inflating: hf_privacy_ner_cleaned/train/dataset_info.json  
  inflating: hf_privacy_ner_cleaned/train/data-00000-of-00001.arrow  
  inflating: hf_privacy_ner_cleaned/train/cache-2f10

In [5]:
pip install evaluate

In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [7]:
from datasets import load_from_disk
from transformers import (
    DebertaV2Tokenizer,
    DebertaV2ForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
import evaluate
import numpy as np

# Load dataset from disk
dataset = load_from_disk("privacy-ner-att/datasets/hf_privacy_ner_cleaned")

# Load tokenizer
tokenizer = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-base")

# Tokenize the dataset
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize, batched=True)

# Rename target column to "labels" for Trainer compatibility
tokenized_dataset = tokenized_dataset.rename_column("privacy_class_label", "labels")

# Set format for PyTorch
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Load pre-trained model with 2 output labels (binary classification)
model = DebertaV2ForSequenceClassification.from_pretrained("microsoft/deberta-v3-base", num_labels=2)

# Load evaluation metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

# Metric computation function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels)["f1"]
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    report_to="none",
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

# Save fine-tuned model and tokenizer
trainer.save_model("deberta_privacy_finetuned")
tokenizer.save_pretrained("deberta_privacy_finetuned")

# ------------------------------------------
# Inference Example (after training)
# ------------------------------------------

# Load the fine-tuned model into a pipeline
clf = pipeline("text-classification", model="deberta_privacy_finetuned", tokenizer="deberta_privacy_finetuned")

# Example prediction
text_example = "Hi, my name is John and I live at 1234 Elm Street."
pred = clf(text_example)

# Print result
print(f"Input: {text_example}")
print(f"Prediction: {pred}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and w

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.054900,0.103795,0.973140,0.983321
2,0.008000,0.096324,0.976033,0.985272
3,0.001800,0.100099,0.978099,0.986552


/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cuda:0


Input: Hi, my name is John and I live at 1234 Elm Street.
Prediction: [{'label': 'LABEL_1', 'score': 0.9999895095825195}]


In [9]:

from transformers import pipeline
from tqdm import tqdm
from sklearn.metrics import classification_report


In [11]:
# Generate predictions on the test set
# test_results = trainer.predict(tokenized_dataset["test"])

# Predicted class labels (as list)
predictions = np.argmax(test_results.predictions, axis=-1).tolist()

# True class labels (as list)
true_labels = test_results.label_ids.tolist()

# Evaluate performance
print(classification_report(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94       454
           1       0.98      0.99      0.99      1966

    accuracy                           0.98      2420
   macro avg       0.97      0.96      0.96      2420
weighted avg       0.98      0.98      0.98      2420

